# Deep Learning Visual MCQ Solver
## Strategy: Vision Language Model (VLM) — Image directly → Answer

**No OCR. No training. No fine-tuning.**

Pipeline: `PNG Image → Qwen2.5-VL-7B → Chain-of-Thought Reasoning → Majority Vote → 1/2/3/4 (or 5 to skip)`

---
### Evaluation Rules:
- **+1** for correct answer
- **-0.25** for incorrect answer
- **0** for skipped (output 5)
- **-1** for hallucinated value (anything other than 1/2/3/4/5)

### Strategy: Only answer when confident. Output 5 to skip uncertain questions.
---

## Cell 1: Install Dependencies

In [ ]:
!pip install -q "transformers>=4.52.0"
!pip install -q accelerate
!pip install -q qwen-vl-utils
!pip install -q Pillow
!pip install -q torchvision
!pip install -q bitsandbytes
!pip install -q sentencepiece
print('All dependencies installed.')

## Cell 2: Imports

In [ ]:
import os, re, time, torch, warnings, json, shutil
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    Qwen2_5_VLProcessor,
    BitsAndBytesConfig
)
from qwen_vl_utils import process_vision_info

print(f'PyTorch        : {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'Number of GPUs : {n_gpus}')
    total_vram = 0
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        total_vram += vram
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} - {vram:.1f} GB')
    print(f'  Total VRAM: {total_vram:.1f} GB')

## Cell 3: Configuration

In [ ]:
# ==============================================================
# PATHS — update BASE_DIR to where competition data is placed
# Folder structure expected:
#   BASE_DIR/
#     images/         <- PNG images
#     test.csv        <- image_id, image_name columns
#     sample_submission.csv
# ==============================================================
BASE_DIR               = './'
TEST_CSV_PATH          = os.path.join(BASE_DIR, 'test.csv')
IMAGE_DIR              = os.path.join(BASE_DIR, 'images')
OUTPUT_PATH            = './submission.csv'
MODEL_DIR              = './model/'  # <-- path to Qwen2.5-VL-7B weights

# ==============================================================
# INFERENCE SETTINGS
# ==============================================================
N_VOTES             = 3      # majority voting runs per image
LLM_TEMP            = 0.5    # temperature for voting runs
MAX_NEW_TOKENS      = 1024   # enough for chain-of-thought
USE_4BIT            = True   # 4-bit quantization
MAX_IMAGE_PIXELS    = 1280 * 28 * 28
CONFIDENCE_THRESHOLD = 2     # min votes needed out of N_VOTES to answer (else output 5)

print('Config loaded.')
print(f'Model path  : {MODEL_DIR}')
print(f'Test CSV    : {TEST_CSV_PATH}')
print(f'Image dir   : {IMAGE_DIR}')
print(f'Output      : {OUTPUT_PATH}')
print(f'N votes     : {N_VOTES}')
print(f'Confidence  : {CONFIDENCE_THRESHOLD}/{N_VOTES} votes needed to answer')

## Cell 4: Fix preprocessor_config and Load Model

In [ ]:
# Fix preprocessor_config.json if needed (Qwen2_5_VLImageProcessor name issue)
pre_path = os.path.join(MODEL_DIR, 'preprocessor_config.json')
if os.path.exists(pre_path):
    with open(pre_path) as f:
        pre_cfg = json.load(f)
    if pre_cfg.get('image_processor_type') == 'Qwen2_5_VLImageProcessor':
        pre_cfg['image_processor_type'] = 'Qwen2VLImageProcessor'
        pre_cfg['processor_class'] = 'Qwen2VLProcessor'
        with open(pre_path, 'w') as f:
            json.dump(pre_cfg, f, indent=2)
        print('Fixed preprocessor_config.json')
    else:
        print('preprocessor_config.json OK')

print('Loading processor...')
processor = Qwen2_5_VLProcessor.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
    min_pixels=256 * 28 * 28,
    max_pixels=MAX_IMAGE_PIXELS
)
print('Processor loaded!')

print('Loading model (2-3 minutes)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_DIR,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    local_files_only=True
)
model.eval()
print('Model loaded!')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        used  = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i} VRAM: {used:.2f} / {total:.1f} GB used')

## Cell 5: Prompt Templates

In [ ]:
MAIN_PROMPT = """You are an expert in deep learning, machine learning, and mathematics.

Look at this image carefully. It contains a multiple choice question with 4 options (1, 2, 3, 4).

Instructions:
1. Read the COMPLETE question carefully
2. Read ALL FOUR options 1, 2, 3, 4 carefully
3. Analyze each option one by one
4. For math questions: show your calculation step by step
5. For theory questions: reason through each option
6. NEVER pick an answer just because it sounds related — verify it is CORRECT
7. Rate your confidence: HIGH / MEDIUM / LOW
8. At the very end, write exactly: FINAL ANSWER: X
   where X is 1, 2, 3, or 4

IMPORTANT REMINDERS:
- For parameter counting: include bias terms unless stated otherwise
- For Conv2D output size: formula = (input - kernel + 2*padding) / stride + 1
- For Hessian/eigenvalue questions: all positive eigenvalues = minimum, mixed = saddle point
- For loss landscape: sharp minima → poor generalization
- Always evaluate ALL 4 options before deciding"""

ALTERNATE_PROMPT = """You are a deep learning professor solving an exam question.

This image shows a multiple choice question. Options are numbered 1, 2, 3, 4.

Solve methodically:
- State what the question is asking
- For each option, explain why it is correct or incorrect
- Show any calculations needed
- State your confidence level: HIGH / MEDIUM / LOW
- End with: FINAL ANSWER: X (where X is 1, 2, 3, or 4)"""

print('Prompts loaded.')

## Cell 6: Image Preprocessing

In [ ]:
from PIL import ImageEnhance, ImageFilter

def preprocess_image(image_path: str) -> Image.Image:
    img = Image.open(image_path).convert('RGB')
    # Upscale small images for better readability
    w, h = img.size
    if w < 800:
        scale = 800 / w
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    # Enhance contrast slightly
    img = ImageEnhance.Contrast(img).enhance(1.2)
    img = ImageEnhance.Sharpness(img).enhance(1.3)
    return img

print('Preprocessing function ready.')

## Cell 7: Answer Parser and VLM Inference

In [ ]:
def parse_answer(text: str) -> str:
    """Extract answer 1/2/3/4 from model response. Returns None if not found."""
    # Look for FINAL ANSWER: X pattern first
    patterns = [
        r'FINAL ANSWER:\s*([1234])',
        r'Final Answer:\s*([1234])',
        r'final answer:\s*([1234])',
        r'The answer is\s*[:\s]*([1234])',
        r'Answer:\s*([1234])\s*$',
        r'^\s*([1234])\s*$',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.MULTILINE | re.IGNORECASE)
        if match:
            return match.group(1)
    # Last resort: find the last standalone 1/2/3/4 in text
    matches = re.findall(r'\b([1234])\b', text)
    if matches:
        return matches[-1]
    return None  # could not parse


def extract_confidence(text: str) -> str:
    """Extract confidence level from model response."""
    text_upper = text.upper()
    if 'HIGH' in text_upper:
        return 'HIGH'
    elif 'MEDIUM' in text_upper:
        return 'MEDIUM'
    elif 'LOW' in text_upper:
        return 'LOW'
    return 'MEDIUM'  # default


def query_model(image_path: str, prompt: str, temperature: float = 0.1) -> str:
    """Run VLM inference on a single image."""
    img = preprocess_image(image_path)
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': prompt}
        ]
    }]
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=temperature,
            do_sample=(temperature > 0),
        )
    generated_ids_trimmed = [
        out[len(inp):]
        for inp, out in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]


def solve_mcq_image(image_path: str, use_voting: bool = True, verbose: bool = True) -> int:
    """
    Main function: takes image path, returns answer as int (1/2/3/4) or 5 (skip).
    Returns 5 if model is not confident enough.
    """
    fname = os.path.basename(image_path)
    if verbose:
        print(f'\nProcessing: {fname}')

    if use_voting:
        # Run N_VOTES times with alternating prompts
        votes = []
        confidences = []
        prompts = [MAIN_PROMPT, ALTERNATE_PROMPT, MAIN_PROMPT]  # alternate
        for i in range(N_VOTES):
            prompt = prompts[i % len(prompts)]
            temp = 0.1 if i == 0 else LLM_TEMP
            response = query_model(image_path, prompt, temperature=temp)
            ans = parse_answer(response)
            conf = extract_confidence(response)
            if ans:
                votes.append(ans)
                confidences.append(conf)
            if verbose:
                print(f'  Vote {i+1}: {ans} (confidence: {conf})')

        if not votes:
            if verbose: print('  No valid answer found → skipping (5)')
            return 5

        most_common, count = Counter(votes).most_common(1)[0]

        # Skip if not enough agreement
        if count < CONFIDENCE_THRESHOLD:
            if verbose: print(f'  Low agreement ({count}/{N_VOTES}) → skipping (5)')
            return 5

        # Skip if majority vote is LOW confidence
        low_conf_count = confidences.count('LOW')
        if low_conf_count > N_VOTES // 2:
            if verbose: print(f'  Low confidence → skipping (5)')
            return 5

        if verbose:
            print(f'  Final answer: {most_common} (votes: {count}/{N_VOTES})')
        return int(most_common)

    else:
        # Single run — always answer (for testing)
        response = query_model(image_path, MAIN_PROMPT, temperature=0.1)
        if verbose:
            print(f'  Full response:\n{response}')
        ans = parse_answer(response)
        if ans is None:
            if verbose: print('  Could not parse answer → returning 5')
            return 5
        if verbose:
            print(f'  Answer: {ans}')
        return int(ans)


print('Inference functions ready.')

## Cell 8: Validate Pipeline on Synthetic Test Images

In [ ]:
def make_test_image(question: str, options: dict, filepath: str):
    img  = Image.new('RGB', (900, 420), color='white')
    draw = ImageDraw.Draw(img)
    try:
        font_q = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 20)
        font_o = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18)
    except:
        font_q = font_o = ImageFont.load_default()

    y = 30
    words, line = question.split(), ''
    for word in words:
        if len(line + word) < 75:
            line += word + ' '
        else:
            draw.text((30, y), line.strip(), fill='black', font=font_q)
            y += 32; line = word + ' '
    draw.text((30, y), line.strip(), fill='black', font=font_q)
    y += 55

    # Options numbered 1/2/3/4 (not A/B/C/D)
    for number, text in options.items():
        draw.text((60, y), f'{number})  {text}', fill=(30, 30, 30), font=font_o)
        y += 42
    img.save(filepath)


os.makedirs('./test_samples', exist_ok=True)

test_cases = [
    {
        'name'    : 'conv_output_size',
        'question': 'What is the output spatial size of a Conv2D with input 32x32, kernel 3x3, padding=0, stride=1?',
        'options' : {'1': '30x30', '2': '32x32', '3': '28x28', '4': '16x16'},
        'answer'  : 1   # (32-3+0)/1 + 1 = 30
    },
    {
        'name'    : 'param_count',
        'question': 'How many trainable parameters does a fully connected layer with 512 inputs and 256 outputs have (including bias)?',
        'options' : {'1': '131072', '2': '131328', '3': '130816', '4': '262144'},
        'answer'  : 2   # 512*256 + 256 = 131328
    },
    {
        'name'    : 'activation_theory',
        'question': 'Which activation function is most prone to the vanishing gradient problem in deep networks?',
        'options' : {'1': 'ReLU', '2': 'Sigmoid', '3': 'Leaky ReLU', '4': 'ELU'},
        'answer'  : 2
    },
    {
        'name'    : 'batch_norm',
        'question': 'During inference time, Batch Normalization uses which statistics?',
        'options' : {'1': 'Current batch mean and variance', '2': 'Running average of mean and variance', '3': 'Global dataset mean and variance', '4': 'Layer output mean only'},
        'answer'  : 2
    }
]

print('Running validation on synthetic test images...\n')
correct = 0
for tc in test_cases:
    path = f"./test_samples/{tc['name']}.png"
    make_test_image(tc['question'], tc['options'], path)
    pred = solve_mcq_image(path, use_voting=False, verbose=True)
    ok   = pred == tc['answer']
    correct += int(ok)
    print(f"  → Predicted={pred} | Expected={tc['answer']} | {'✅ CORRECT' if ok else '❌ WRONG'}\n")

print(f'Validation accuracy: {correct}/{len(test_cases)} = {100*correct/len(test_cases):.0f}%')

## Cell 9: Test With Your Own Image

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Put your image path here
my_image_path = './your_question.png'  # <-- change this

if os.path.exists(my_image_path):
    img = mpimg.imread(my_image_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Your Question Image')
    plt.show()

    print('Running model...')
    pred = solve_mcq_image(my_image_path, use_voting=False, verbose=True)
    print(f'\n✅ Model Answer: {pred}')
else:
    print(f'Image not found: {my_image_path}')
    print('Place your image file in the same folder and update my_image_path')

## Cell 10: Full Inference on Competition Test Set

In [ ]:
# Read test.csv
if os.path.exists(TEST_CSV_PATH):
    test_df = pd.read_csv(TEST_CSV_PATH)
    print(f'Loaded test.csv: {test_df.shape}')
    print(f'Columns: {test_df.columns.tolist()}')
    print(test_df.head(3))
else:
    # Fallback: scan images folder directly
    img_files = sorted(Path(IMAGE_DIR).glob('*.png'))
    if not img_files:
        img_files = sorted(Path(IMAGE_DIR).glob('*.jpg'))
    test_df = pd.DataFrame({
        'image_id'  : [f.stem for f in img_files],
        'image_name': [f.stem for f in img_files]
    })
    print(f'Built template for {len(test_df)} images (no test.csv found)')

total = len(test_df)
est_mins = total * 25 * N_VOTES / 60
print(f'\nTotal questions : {total}')
print(f'Estimated time  : ~{est_mins:.0f} minutes with N_VOTES={N_VOTES}')

predictions = []
failed_ids  = []
start_time  = time.time()

for idx, row in test_df.iterrows():
    image_id   = str(row['image_id'])
    image_name = str(row['image_name'])

    # Try multiple path formats
    candidates = [
        os.path.join(IMAGE_DIR, f'{image_name}.png'),
        os.path.join(IMAGE_DIR, f'{image_name}.jpg'),
        os.path.join(IMAGE_DIR, image_name),
        os.path.join(IMAGE_DIR, f'{image_name}.PNG'),
    ]
    img_path = next((p for p in candidates if os.path.exists(p)), None)

    if img_path is None:
        print(f'  [{idx+1}/{total}] NOT FOUND: {image_name} → skipping (5)')
        predictions.append(5)
        failed_ids.append(image_id)
        continue

    pred = solve_mcq_image(img_path, use_voting=True, verbose=True)
    predictions.append(pred)

    elapsed = time.time() - start_time
    avg     = elapsed / (idx + 1)
    eta     = avg * (total - idx - 1)
    print(f'  [{idx+1}/{total}] {image_name} → {pred} | Elapsed: {elapsed/60:.1f}m | ETA: {eta/60:.1f}m')

# Build submission CSV in required format
submission_df = pd.DataFrame({
    'id'        : test_df['image_id'],
    'image_name': test_df['image_name'],
    'option'    : predictions
})
submission_df.to_csv(OUTPUT_PATH, index=False)

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f'Done in {total_time/60:.1f} minutes')
print(f'Total predictions  : {len(predictions)}')
print(f'Failed/skipped     : {predictions.count(5)}')
print(f'Answer distribution: {dict(Counter(predictions))}')
print(f'Saved to           : {OUTPUT_PATH}')
print(f"{'='*60}")

## Cell 11: Verify Submission

In [ ]:
final = pd.read_csv(OUTPUT_PATH)
print(f'Shape   : {final.shape}')
print(f'Columns : {final.columns.tolist()}')
print(f'\nAnswer distribution:')
print(final['option'].value_counts())
print(f'\nFirst 10 rows:')
print(final.head(10))

# Sanity checks
valid_values = {1, 2, 3, 4, 5}
invalid = final[~final['option'].isin(valid_values)]
if len(invalid) > 0:
    print(f'\n⚠️  WARNING: {len(invalid)} invalid answers (will be treated as hallucinated = -1 each!):')
    print(invalid)
else:
    skipped = (final['option'] == 5).sum()
    answered = len(final) - skipped
    print(f'\n✅ All {len(final)} answers valid!')
    print(f'   Answered : {answered}')
    print(f'   Skipped  : {skipped}')
    print(f'\nReady to submit!')